# Proximal Policy Optimization (PPO)

**Proximal Policy Optimization (PPO)** adalah algoritma Policy Gradient yang banyak digunakan dalam Reinforcement Learning, termasuk pada pelatihan RLHF.

PPO mengatasi salah satu masalah utama Policy Gradient standar seperti **REINFORCE**, yaitu **stabilitas pelatihan**.

Pada REINFORCE, jika update bobot Neural Network terlalu besar dalam satu step, performa policy dapat mendadak menurun (*policy collapse*).

PPO mencegah hal tersebut dengan membatasi (*clipping*) seberapa jauh kebijakan baru boleh berubah dari kebijakan lama.

## 3 Konsep Utama PPO

### 1. Clipping Objective (PPO-Clip)

PPO menghitung rasio probabilitas kebijakan baru dibandingkan kebijakan lama:

$$
r_t(\theta) =
\frac{\pi_\theta(a_t \mid s_t)}
{\pi_{\theta_{\text{old}}}(a_t \mid s_t)}
$$

Jika rasio terlalu besar atau terlalu kecil, yaitu berada di luar rentang:

$$
1 - \epsilon \leq r_t(\theta) \leq 1 + \epsilon
$$

maka nilai objective akan **di-clipping**.

Biasanya:

$$
\epsilon = 0.2
$$

Tujuannya adalah mencegah perubahan policy yang terlalu ekstrem dalam satu update.

### 2. Advantage Function

PPO menggunakan **Advantage Function** untuk mengukur seberapa baik suatu aksi dibandingkan nilai rata-rata pada state tersebut.

$$
A(s, a) = Q(s, a) - V(s)
$$

* `Q(s, a)` → nilai melakukan aksi `a` pada state `s`.
* `V(s)` → nilai rata-rata dari state `s`.
* `A(s, a)` → seberapa baik aksi tersebut dibandingkan kondisi rata-rata.

Jika `A` positif, aksi tersebut lebih baik dari rata-rata.

Jika `A` negatif, aksi tersebut lebih buruk dari rata-rata.

### 3. Arsitektur Actor-Critic

PPO menggunakan pendekatan **Actor-Critic** yang terdiri dari dua komponen:

* **Actor** → memprediksi probabilitas aksi atau Policy Network.
* **Critic** → memprediksi nilai state `V(s)` atau Value Network yang digunakan untuk menghitung Advantage.

Secara sederhana:

```text
State
  ↓
 ┌───────────────┐
 │ Actor         │ → Probabilitas Action
 │ Critic        │ → Value V(s)
 └───────────────┘
          ↓
    Advantage A(s,a)
          ↓
    PPO Clipping
          ↓
    Update Policy
```


In [3]:
!pip install stable-baselines3[extra] gymnasium -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 11.6 MB/s eta 0:00:00


In [4]:
import gymnasium as env_lib
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

print("Import berhasil! Library siap digunakan.")

# 1. Buat Environment
env = env_lib.make("CartPole-v1")

# 2. Inisialisasi Model PPO
# 'MlpPolicy' berarti menggunakan Multi-Layer Perceptron (Neural Network standar)
model = PPO(
    policy="MlpPolicy",
    env=env,
    learning_rate=3e-4,
    n_steps=2048,           # Jumlah langkah yang dikumpulkan sebelum update
    batch_size=64,
    n_epochs=10,            # Jumlah epoch pembaruan per batch
    gamma=0.99,
    clip_range=0.2,         # Epsilon clipping (1 - 0.2 s.d 1 + 0.2)
    verbose=1
)

# 3. Latih Agen
print("--- Memulai Pelatihan PPO ---")
model.learn(total_timesteps=25000)

# 4. Evaluasi Performa Model
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10)
print(f"\nRata-rata Reward setelah training: {mean_reward:.2f} +/- {std_reward:.2f}")

# 5. Simpan Model
model.save("ppo_cartpole_model")
env.close()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Import berhasil! Library siap digunakan.
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


--- Memulai Pelatihan PPO ---


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 23.2     |
|    ep_rew_mean     | 23.2     |
| time/              |          |
|    fps             | 1158     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 25.7        |
|    ep_rew_mean          | 25.7        |
| time/                   |             |
|    fps                  | 731         |
|    iterations           | 2           |
|    time_elapsed         | 5           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.007922377 |
|    clip_fraction        | 0.0999      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.686      |
|    explained_variance   | -0.00335    |
|    learning_rate        | 0.

/usr/local/lib/python3.13/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(



Rata-rata Reward setelah training: 500.00 +/- 0.00


In [5]:
# 1. Buat Environment
env = env_lib.make("CartPole-v1")

# 2. Inisialisasi Model PPO
model = PPO("MlpPolicy", env, verbose=1)

# 3. Latih Agen
model.learn(total_timesteps=10000)

# 4. Evaluasi
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10)
print(f"Rata-rata Reward: {mean_reward:.2f} +/- {std_reward:.2f}")

env.close()

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 23.3     |
|    ep_rew_mean     | 23.3     |
| time/              |          |
|    fps             | 1144     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 27.5        |
|    ep_rew_mean          | 27.5        |
| time/                   |             |
|    fps                  | 567         |
|    iterations           | 2           |
|    time_elapsed         | 7           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.008500181 |
|    clip_fraction        | 0.118       |
|    clip_range           | 0.2         |
|    entropy_loss   